In [1]:
import pandas as pd
import polars as pl
import duckdb
from datetime import datetime

# Moving Data To Parquet

In [2]:
CSV_FILE_PATH = "../2022_place_canvas_history.csv"
PARQUET_FILE_PATH = "../2022_place_canvas_history.parquet"

START_DATE = datetime(2022, 4, 2, 12)
END_DATE = datetime(2022, 4, 2, 16)

In [50]:
df = pl.scan_csv(CSV_FILE_PATH).with_columns(
    pl.coalesce(
        pl.col("timestamp").str.strptime(
            pl.Datetime, "%Y-%m-%d %H:%M:%S.%f %Z", strict=False
        ),
        pl.col("timestamp").str.strptime(
            pl.Datetime, "%Y-%m-%d %H:%M:%S %Z", strict=False
        )
    )
    .dt.replace_time_zone(None)
    .alias("timestamp")
).drop("user_id")

df.sink_parquet(PARQUET_FILE_PATH)

/var/folders/ml/fzp4hsz17tb4xtns43992l0w0000gn/T/ipykernel_67469/429030234.py:3: ChronoFormatWarning: Detected the pattern `.%f` in the chrono format string. This pattern should not be used to parse values after a decimal point. Use `%.f` instead. See the full specification: https://docs.rs/chrono/latest/chrono/format/strftime
  pl.col("timestamp").str.strptime(


# DuckDB

In [18]:
## Most Frequent Color
query = f"""
    SELECT 
        pixel_color,
        COUNT(*) AS frequency
    FROM
        '{PARQUET_FILE_PATH}'
    WHERE
        timestamp >= '{START_DATE}' AND
        timestamp <= '{END_DATE}'
    GROUP BY
        pixel_color
    ORDER BY
        frequency DESC
    LIMIT 5
"""

duckdb.query(query)

┌─────────────┬───────────┐
│ pixel_color │ frequency │
│   varchar   │   int64   │
├─────────────┼───────────┤
│ #000000     │   1094641 │
│ #FFFFFF     │    837610 │
│ #FF4500     │    794903 │
│ #2450A4     │    403326 │
│ #FFD635     │    297289 │
└─────────────┴───────────┘

In [19]:
## Most Frequent Coordinate
query = f"""
    SELECT
        coordinate,
        COUNT(*) AS frequency
    FROM
        '{PARQUET_FILE_PATH}'
    WHERE
        timestamp >= '{START_DATE}' AND
        timestamp <= '{END_DATE}'
    GROUP BY
        coordinate
    ORDER BY
        frequency DESC
    LIMIT 5
"""

duckdb.query(query)

┌────────────┬───────────┐
│ coordinate │ frequency │
│  varchar   │   int64   │
├────────────┼───────────┤
│ 859,766    │      7529 │
│ 860,766    │      7520 │
│ 359,564    │      3491 │
│ 0,0        │      2877 │
│ 999,999    │      2572 │
└────────────┴───────────┘

# Polars

In [39]:
df = pl.read_parquet(PARQUET_FILE_PATH)

In [40]:
pl_color = df.filter(
    pl.col("timestamp").is_between(START_DATE, END_DATE)
    ).group_by("pixel_color").len().sort(by="len", descending=True).limit(5)

pl_coord = df.filter(
    pl.col("timestamp").is_between(START_DATE, END_DATE)
    ).group_by("coordinate").len().sort(by="len", descending=True).limit(5)

print(pl_color, pl_coord)

shape: (5, 2)
┌─────────────┬─────────┐
│ pixel_color ┆ len     │
│ ---         ┆ ---     │
│ str         ┆ u32     │
╞═════════════╪═════════╡
│ #000000     ┆ 1094641 │
│ #FFFFFF     ┆ 837610  │
│ #FF4500     ┆ 794903  │
│ #2450A4     ┆ 403326  │
│ #FFD635     ┆ 297289  │
└─────────────┴─────────┘ shape: (5, 2)
┌────────────┬──────┐
│ coordinate ┆ len  │
│ ---        ┆ ---  │
│ str        ┆ u32  │
╞════════════╪══════╡
│ 859,766    ┆ 7529 │
│ 860,766    ┆ 7520 │
│ 359,564    ┆ 3491 │
│ 0,0        ┆ 2877 │
│ 999,999    ┆ 2572 │
└────────────┴──────┘


# Pandas

In [3]:
df = pd.read_parquet(PARQUET_FILE_PATH)

In [ ]:
pd_color = (df[(df["timestamp"] >= START_DATE) & (df["timestamp"] <= END_DATE)]).groupby("pixel_color").count()

# pd_coord = (df[(df["timestamp"] >= START_DATE) & (df["timestamp"] <= END_DATE)]).groupby("pixel_color").count()
pd_color

KeyError: 'count'